In [ ]:
from pymol import cmd
import shutil
import os
import glob

input_folder = "path/to/input_files"


In [ ]:
output_folder = "superposed_output"
os.makedirs(output_folder, exist_ok=True)

reference_file = "path/to/reference.pdb"

input_files = glob.glob(os.path.join(input_folder, "*.pdb"))
print(f"Found {len(input_files)} files")

thresholds = [2.0, 3.0, 5.0, 8.0]
results = {t: [] for t in thresholds}

# Chain of the query object that is checked for clashes
QUERY_CHAIN = "B"

# Part of the reference excluded from the clash check
excluded_ref_sel = "chain A and resi 1-20+100-120"

# Part of the reference used for the superposition
alignment_sel = "chain A and resi 5-30"

for input_file in input_files:
    basename = os.path.splitext(os.path.basename(input_file))[0]

    cmd.reinitialize()

    output_file = os.path.join(output_folder, f"{basename}_superposed.pdb")
    shutil.copy(input_file, output_file)

    cmd.load(output_file, "query")
    cmd.load(reference_file, "reference")

    result = cmd.super("query", f"reference and {alignment_sel}")
    print(f"{basename}: RMSD {result[0]:.3f} Å over {result[1]} atoms")

    # Clash check: query chain vs. the whole reference except the excluded region
    clash_counts = {}
    for t in thresholds:
        cmd.select("clashes", f"query and chain {QUERY_CHAIN} within {t} of (reference and not ({excluded_ref_sel}))")
        clash_counts[t] = cmd.count_atoms("clashes")

    print("  Clashes -> " + " | ".join([f"{t}Å: {clash_counts[t]}" for t in thresholds]))

    # --- Coloring for visual inspection ---
    cmd.color("grey80", "reference")
    cmd.color("orange", f"reference and ({excluded_ref_sel})")
    cmd.color("skyblue", f"query and chain {QUERY_CHAIN}")

    for t in thresholds:
        if clash_counts[t] == 0:
            results[t].append({
                "file": basename,
                "rmsd": round(result[0], 3),
                "clashes": 0
            })

    cmd.save(output_file)

# Summary per threshold
print("\n--- Summary ---")
for t in thresholds:
    passed = [r["file"] for r in results[t]]
    print(f"\nThreshold {t}Å — {len(passed)} files passed:")
    for f in passed:
        print(f"  - {f}")

print(f"\nDone! All outputs saved to: {output_folder}")
